# Lecture 14: Dolbeault Theory

**Source span.** Printed pages 76-80; physical PDF pages 90-94 in `Lectures on Symplectic Geometry.pdf`. I inspected the local PDF text for this span before revising the notebook.

**Lecture goal.** Explain how an almost complex structure splits complexified tangent and cotangent spaces, how forms acquire type `(l,m)`, and how `partial`, `dbar`, holomorphic functions, and Dolbeault cohomology fit into one bigraded picture.

The source span is a bookkeeping lecture with real geometric consequences. The bookkeeping starts from the two eigenspaces of `J` on `TM tensor C`, passes to the dual splitting of complex-valued one-forms, extends to wedge powers, and then projects the exterior derivative into type components. On functions, the central test is small and powerful: `f` is `J`-holomorphic exactly when the `(0,1)` part of `df`, namely `dbar f`, vanishes.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib
from utils.symplectic import standard_omega

ARTIFACT_TOPIC = "lecture-14"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| complexified tangent bundle | real vector space with complex coefficients | `J` has eigenvalues `+i` and `-i` after complexification |
| `T^{1,0}` and `T^{0,1}` | projection matrices `P10=(I-iJ)/2`, `P01=(I+iJ)/2` | projections are idempotent, complementary, and satisfy eigenvalue equations |
| type decomposition | grid of spaces `Omega^{l,m}` | total degree is `l+m` and arrows change one index |
| partial and dbar | projected exterior derivative pieces | `partial` raises `l`; `dbar` raises `m` |
| J-holomorphic function | symbolic Wirtinger derivative | `dbar f=0` |
| Dolbeault cohomology | chain complex under `dbar` | `dbar^2=0` when the complex splitting is integrable |
| Nijenhuis tensor | obstruction in the homework | vanishing is the integrability test named in Newlander-Nirenberg |

**Library routing.** `numpy` handles projection matrices and eigen-residuals for the splitting. `matplotlib` and `networkx` render the bigraded operator lattice. `sympy` checks `partial` and `dbar` on functions without treating handwritten algebra as a black box.

## Visualization Storyboard

1. **Complexified splitting.** Build `P10` and `P01` from `J`, verify they are complementary projections, and show the `+i/-i` eigenspace split.
2. **Bigraded form lattice.** Draw `Omega^{l,m}` nodes and arrows for `partial` and `dbar`; this is the proof diagram behind `d=partial+dbar` and `dbar^2=0`.
3. **Symbolic holomorphic-function checks.** Use Wirtinger derivatives to test functions killed by `dbar`, functions killed by `partial`, and mixed functions that fail both.
4. **Integrability ledger.** Record the Nijenhuis obstruction and the Newlander-Nirenberg bridge to the next lecture, without importing the homework as copied exercise text.

In [ ]:
# Splitting TM tensor C into +/- i eigenspaces for the standard J on R2.
J = np.array([[0.0, -1.0], [1.0, 0.0]])
I2 = np.eye(2, dtype=complex)
P10 = 0.5 * (I2 - 1j * J)
P01 = 0.5 * (I2 + 1j * J)
projection_residuals = {
    "P10_idempotent": float(np.linalg.norm(P10 @ P10 - P10)),
    "P01_idempotent": float(np.linalg.norm(P01 @ P01 - P01)),
    "complementary_sum": float(np.linalg.norm(P10 + P01 - I2)),
    "orthogonal_product": float(np.linalg.norm(P10 @ P01)),
    "P10_eigen_residual": float(np.linalg.norm(J.astype(complex) @ P10 - 1j * P10)),
    "P01_eigen_residual": float(np.linalg.norm(J.astype(complex) @ P01 + 1j * P01)),
}
assert max(projection_residuals.values()) < 1e-12

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
for ax, matrix, title in [(axes[0], J, "J"), (axes[1], np.abs(P10), "|P10|"), (axes[2], np.abs(P01), "|P01|")]:
    im = ax.imshow(matrix, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([0, 1], labels=["dx", "dy"])
    ax.set_yticks([0, 1], labels=["dx", "dy"])
    for (i, j), value in np.ndenumerate(matrix):
        display_value = abs(value) if np.iscomplexobj(matrix) else value
        if abs(display_value) > 1e-9:
            ax.text(j, i, f"{display_value:.1f}", ha="center", va="center", color="white", fontsize=9)
fig.colorbar(im, ax=axes, fraction=0.03)
fig.suptitle("Projection formulas split the complexified tangent space into T10 and T01")
splitting_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "complexified-tangent-splitting-projections.png")
plt.close(fig)
display_artifact(splitting_path, width=760)
print(projection_residuals)

In [ ]:
# Bigraded form lattice with partial and dbar arrows.
max_degree = 3
G = nx.DiGraph()
for l in range(max_degree + 1):
    for m in range(max_degree + 1 - l):
        G.add_node((l, m), label=f"Omega^{l},{m}")
        if l + 1 + m <= max_degree:
            G.add_edge((l, m), (l + 1, m), op="partial")
        if l + m + 1 <= max_degree:
            G.add_edge((l, m), (l, m + 1), op="dbar")
pos = {(l, m): (l, -m) for l, m in G.nodes}
edge_colors = ["#1b9e77" if data["op"] == "partial" else "#d95f02" for _, _, data in G.edges(data=True)]
edge_labels = {(u, v): data["op"] for u, v, data in G.edges(data=True) if u[0] + u[1] < 2}

fig, ax = plt.subplots(figsize=(9, 6.2))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color=edge_colors, width=1.5)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#f0f0f0", node_size=1600, edgecolors="0.25")
nx.draw_networkx_labels(G, pos, ax=ax, labels={node: f"Omega^{node[0]},{node[1]}" for node in G.nodes}, font_size=9)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, ax=ax)
ax.set_axis_off()
ax.set_title("Bigraded form lattice: partial moves east, dbar moves south")
lattice_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "bigraded-form-lattice-partial-dbar.png")
plt.close(fig)
display_artifact(lattice_path, width=760)
lattice_checks = {"nodes": G.number_of_nodes(), "edges": G.number_of_edges(), "dbar_arrows": sum(1 for _, _, d in G.edges(data=True) if d["op"] == "dbar")}
assert lattice_checks["nodes"] == 10
assert lattice_checks["dbar_arrows"] > 0
print(lattice_checks)

In [ ]:
# Symbolic Wirtinger checks for J-holomorphic and anti-holomorphic functions.
x, y = sp.symbols("x y", real=True)
ii = sp.I
z = x + ii*y
zbar = x - ii*y

def partial_expr(f):
    return sp.simplify(sp.Rational(1, 2) * (sp.diff(f, x) - ii * sp.diff(f, y)))

def dbar_expr(f):
    return sp.simplify(sp.Rational(1, 2) * (sp.diff(f, x) + ii * sp.diff(f, y)))

functions = {
    "z_squared": z**2,
    "zbar_squared": zbar**2,
    "mixed_z_zbar": z * zbar,
    "constant": sp.Integer(3),
}
wirtinger = {
    name: {
        "partial": str(partial_expr(expr)),
        "dbar": str(dbar_expr(expr)),
        "dbar_zero": bool(sp.simplify(dbar_expr(expr)) == 0),
        "partial_zero": bool(sp.simplify(partial_expr(expr)) == 0),
    }
    for name, expr in functions.items()
}
assert wirtinger["z_squared"]["dbar_zero"]
assert wirtinger["zbar_squared"]["partial_zero"]
assert not wirtinger["mixed_z_zbar"]["dbar_zero"]
assert wirtinger["constant"]["dbar_zero"] and wirtinger["constant"]["partial_zero"]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
names = list(functions.keys())
dbar_values = [0 if wirtinger[name]["dbar_zero"] else 1 for name in names]
partial_values = [0 if wirtinger[name]["partial_zero"] else 1 for name in names]
axes[0].bar(names, dbar_values, color="#d95f02")
axes[0].set_title("dbar obstruction: 0 means holomorphic")
axes[0].set_ylabel("nonzero flag")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(names, partial_values, color="#1b9e77")
axes[1].set_title("partial obstruction: 0 means anti-holomorphic")
axes[1].tick_params(axis="x", rotation=25)
fig.suptitle("J-holomorphic functions are exactly the functions killed by dbar")
wirtinger_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "wirtinger-holomorphic-dbar-checks.png")
plt.close(fig)
display_artifact(wirtinger_path, width=760)
print(wirtinger)

In [ ]:
# Integrability and Dolbeault complex ledger.
H = nx.DiGraph()
H.add_edges_from([
    ("almost complex J", "T10 plus T01 splitting"),
    ("T10 plus T01 splitting", "forms decompose by type (l,m)"),
    ("forms decompose by type (l,m)", "project d into partial and dbar pieces"),
    ("project d into partial and dbar pieces", "if d=partial+dbar then d^2 separates by type"),
    ("if d=partial+dbar then d^2 separates by type", "dbar^2=0"),
    ("dbar^2=0", "Dolbeault cohomology"),
    ("Nijenhuis tensor N", "integrability obstruction"),
    ("integrability obstruction", "Newlander-Nirenberg: N=0 iff complex charts"),
    ("complex charts", "d=partial+dbar cleanly"),
    ("J-holomorphic functions", "dbar f=0"),
])
pos = {
    "almost complex J": (0, 1.6),
    "T10 plus T01 splitting": (1.8, 1.6),
    "forms decompose by type (l,m)": (3.6, 1.6),
    "project d into partial and dbar pieces": (5.4, 1.6),
    "if d=partial+dbar then d^2 separates by type": (7.2, 1.6),
    "dbar^2=0": (9.0, 1.6),
    "Dolbeault cohomology": (10.8, 1.6),
    "Nijenhuis tensor N": (3.6, 0.1),
    "integrability obstruction": (5.4, 0.1),
    "Newlander-Nirenberg: N=0 iff complex charts": (7.2, 0.1),
    "complex charts": (9.0, 0.1),
    "d=partial+dbar cleanly": (10.8, 0.1),
    "J-holomorphic functions": (7.2, -1.2),
    "dbar f=0": (9.0, -1.2),
}
fig, ax = plt.subplots(figsize=(14, 5.0))
nx.draw_networkx_edges(H, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="0.35")
nx.draw_networkx_nodes(H, pos, ax=ax, node_color="#c7e9c0", node_size=1900, edgecolors="0.25")
nx.draw_networkx_labels(H, pos, ax=ax, font_size=7)
ax.set_axis_off()
ax.set_title("Dolbeault cohomology requires the type splitting and the integrability gate")
integrability_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "dolbeault-integrability-nijenhuis-ledger.png")
plt.close(fig)
display_artifact(integrability_path, width=760)
integrability_checks = {"ledger_nodes": H.number_of_nodes(), "ledger_edges": H.number_of_edges(), "mentions_nijenhuis": "Nijenhuis tensor N" in H.nodes}
assert integrability_checks["mentions_nijenhuis"]
print(integrability_checks)

## Reading The Visuals

The splitting figure is the algebraic core: `P10=(I-iJ)/2` and `P01=(I+iJ)/2` are complementary projections. Their eigen-residuals show that the image of one is the `+i` eigenspace and the image of the other is the `-i` eigenspace.

The bigraded lattice is the map of all form types. `partial` moves from `(l,m)` to `(l+1,m)`, while `dbar` moves from `(l,m)` to `(l,m+1)`. When the clean two-term splitting of `d` is available and `d^2=0` separates by type, `dbar^2=0` gives the Dolbeault complex.

The symbolic checks make the holomorphic-function definitions executable. Polynomials in `z` are killed by `dbar`; polynomials in `zbar` are killed by `partial`; mixed functions show a nonzero obstruction. The final graph records the integrability warning: the Nijenhuis tensor is the obstruction named in the homework and points toward the next lecture's integrability tests.

A useful way to read the chapter is to separate two levels of structure. The projections and type labels are pointwise linear algebra attached to any almost complex structure; the Dolbeault complex is a stronger statement that those types interact with exterior differentiation in a chart-compatible way. That distinction keeps the symbolic examples honest: they demonstrate the familiar integrable model, while the ledger marks exactly where the general almost-complex case still needs a hypothesis.

In [ ]:
residuals = {
    **projection_residuals,
    "lattice_nodes": lattice_checks["nodes"],
    "lattice_edges": lattice_checks["edges"],
    "dbar_arrows": lattice_checks["dbar_arrows"],
    "z_squared_dbar_zero": wirtinger["z_squared"]["dbar_zero"],
    "zbar_squared_partial_zero": wirtinger["zbar_squared"]["partial_zero"],
    "mixed_function_dbar_nonzero": not wirtinger["mixed_z_zbar"]["dbar_zero"],
    "integrability_ledger_nodes": integrability_checks["ledger_nodes"],
    "mentions_nijenhuis": integrability_checks["mentions_nijenhuis"],
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "dolbeault-residuals.json")

source_span = {
    "title": "Dolbeault Theory",
    "printed_pages": "76-80",
    "physical_pdf_pages": "90-94",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Splittings",
        "Forms of type (l,m)",
        "J-holomorphic functions",
        "Dolbeault cohomology",
        "Homework 10 integrability themes",
    ],
    "concepts": [
        "type decomposition",
        "partial and dbar",
        "J-holomorphic function",
        "Dolbeault cohomology",
        "Nijenhuis tensor",
        "integrability",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Make Dolbeault type decomposition visible as projection algebra, a bigraded lattice, and symbolic dbar checks.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "type decomposition", "representation": "projection matrices and eigen-residuals", "library": "numpy + matplotlib", "why": "the splitting is defined by eigenspaces of J"},
        {"concept": "partial and dbar", "representation": "bigraded operator lattice", "library": "networkx + matplotlib", "why": "operator arrows are the clearest view of the type shifts"},
        {"concept": "J-holomorphic function", "representation": "Wirtinger symbolic checks", "library": "sympy", "why": "dbar f=0 is an exact derivative identity on examples"},
        {"concept": "Dolbeault cohomology", "representation": "integrability dependency graph", "library": "networkx", "why": "dbar cohomology depends on the differential-complex condition"},
    ],
    "visual_sequence": [
        {"concept": "complexified splitting", "artifact": "artifacts/lecture-14/figures/complexified-tangent-splitting-projections.png", "inspection_target": "P10 and P01 are complementary eigenspace projections"},
        {"concept": "bigraded form lattice with operator arrows", "artifact": "artifacts/lecture-14/figures/bigraded-form-lattice-partial-dbar.png", "inspection_target": "partial and dbar shift different type indices"},
        {"concept": "J-holomorphic dbar check", "artifact": "artifacts/lecture-14/figures/wirtinger-holomorphic-dbar-checks.png", "inspection_target": "dbar kills z-polynomials and detects mixed terms"},
        {"concept": "Dolbeault integrability ledger", "artifact": "artifacts/lecture-14/figures/dolbeault-integrability-nijenhuis-ledger.png", "inspection_target": "Nijenhuis vanishing is the gate to complex charts and Dolbeault complexes"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(splitting_path.relative_to(BOOK_ROOT)),
        str(lattice_path.relative_to(BOOK_ROOT)),
        str(wirtinger_path.relative_to(BOOK_ROOT)),
        str(integrability_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "projections_are_complementary": bool(max(projection_residuals.values()) < 1e-12),
        "lattice_contains_dbar_complex_arrows": bool(lattice_checks["dbar_arrows"] > 0),
        "holomorphic_function_killed_by_dbar": bool(wirtinger["z_squared"]["dbar_zero"]),
        "antiholomorphic_function_killed_by_partial": bool(wirtinger["zbar_squared"]["partial_zero"]),
        "mixed_function_detected_by_dbar": bool(not wirtinger["mixed_z_zbar"]["dbar_zero"]),
        "integrability_ledger_mentions_nijenhuis": bool(integrability_checks["mentions_nijenhuis"]),
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 14 checks passed; artifacts are present and nonempty.")

## Takeaways

- Complexifying tangent spaces lets `J` split vectors into `T10` and `T01` eigenspaces.
- Complex-valued forms split by type `(l,m)`, and the exterior derivative can be projected into type-changing pieces.
- On functions, `dbar f=0` is the executable holomorphicity test; `partial f=0` is the anti-holomorphic counterpart.
- Dolbeault cohomology uses `dbar` as a differential, so `dbar^2=0` is the structural condition to watch.
- The Nijenhuis tensor is the integrability obstruction named by the homework and leads directly to the next lecture's complex-manifold tests.